In [2]:
import pandas as pd
import os

In [3]:
os.chdir("/Users/alexanderhannah/Desktop/Github-projects/airr-ml-25")
os.getcwd()

'/Users/alexanderhannah/Desktop/Github-projects/airr-ml-25'

In [4]:
print(os.getcwd())
for root, dirs, files in os.walk("data"):
    level = root.replace("data", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 5:  # limit depth so it doesn't print every .tsv file
        subindent = " " * 2 * (level + 1)
        for file in files[:3]:  # show just first 3 files as examples
            print(f"{subindent}{file}")

/Users/alexanderhannah/Desktop/Github-projects/airr-ml-25
data/
  adaptive-immune-profiling-challenge-2025.zip
  sample_submissions.csv
  test_datasets/
    test_datasets/
      test_dataset_8_2/
        9668c8f2846bd137f59a98bce0276e69.tsv
        6db14a47f1970b52e9105b9544d2a27a.tsv
        7fc452af29e18a7d809f8a6d400d8da6.tsv
      test_dataset_1/
        fb949f48f5710332ea721058d47540f2.tsv
        0e0bad32b85c4d81fae8b97c65752108.tsv
        bf65c30e0a3d5d19c10976d6c1b4b303.tsv
      test_dataset_6/
        226d1ae0b21dd898e288c46072e252a0.tsv
        78dbc16e30501c501e996dfdba988801.tsv
        318b60dbd6194271c1b0f7a50e379634.tsv
      test_dataset_8_3/
        c9c67aa5dbaa8d7a5afde3482e284f6d.tsv
        36d39945c12307ea6b3f59ff7fc29c19.tsv
        c6dbc8cd876a6d3f3d6cffdbe12d4a84.tsv
      test_dataset_7_1/
        90bbfead56eb7124bc695611eff215fa.tsv
        ff79aff0790269e90204eeea35eb2cb6.tsv
        1f8de89cadf315921c91ab20726b3984.tsv
      test_dataset_5/
        5397df7

In [5]:
# Load metadata for one data set
base_path = "data/train_datasets/train_datasets/train_dataset_1"
metadata = pd.read_csv(f"{base_path}/metadata.csv")
print(metadata.shape)
metadata.head()

(400, 3)


,repertoire_id,filename,label_positive
0,44967b361684556629a8b61288daf20c,44967b361684556629a8b61288daf20c.tsv,True
1,36c146e43178cc60d3694117e85cf8f0,36c146e43178cc60d3694117e85cf8f0.tsv,False
2,90e072812957e184a00d7dd6e6fc1113,90e072812957e184a00d7dd6e6fc1113.tsv,False
3,e067bc70fd5eb332289ab527ee07b10e,e067bc70fd5eb332289ab527ee07b10e.tsv,True
4,e23d10263fd2069cdf5d7033532d11ad,e23d10263fd2069cdf5d7033532d11ad.tsv,False


In [6]:
# Load sequences for one person
sample_file = metadata.iloc[0]["filename"]
sequences = pd.read_csv(f"{base_path}/{sample_file}", sep="\t")
print(sequences.shape)
sequences.head()

(25000, 3)


,junction_aa,v_call,j_call
0,CASSQIPELDLLLDTQYF,TRBV3-1,TRBJ2-3
1,CASSLGGFGGNEQFF,TRBV7-6,TRBJ2-1
2,CSASPGPDEQFF,TRBV20-1,TRBJ2-1
3,CASIHRRGVGEQFF,TRBV27,TRBJ2-1
4,CASSPPRGSTGNTIYF,TRBV4-1,TRBJ1-3


In [7]:
# How many unique v_call and j_call values are there?
print("Unique v_calls:", sequences["v_call"].nunique())
print("Unique j_calls:", sequences["j_call"].nunique())

# How long are the junction_aa sequences?
sequences["junction_length"] = sequences["junction_aa"].str.len()
print("\nJunction length stats:")
print(sequences["junction_length"].describe())

Unique v_calls: 54
Unique j_calls: 13

Junction length stats:
count    25000.000000
mean        14.869160
std          1.822178
min          6.000000
25%         14.000000
50%         15.000000
75%         16.000000
max         24.000000
Name: junction_length, dtype: float64


In [8]:
# Check label balance
print(metadata["label_positive"].value_counts())

label_positive
True     200
False    200
Name: count, dtype: int64


In [9]:
# Check v_call and j_call vocabulary across all datasets
import glob

all_v_calls = set()
all_j_calls = set()

for dataset in range(1, 9):
    meta_path = f"data/train_datasets/train_datasets/train_dataset_{dataset}/metadata.csv"
    meta = pd.read_csv(meta_path)
    
    # sample just 5 repertoires per dataset to keep it fast
    for filename in meta["filename"].values[:5]:
        seq_path = f"data/train_datasets/train_datasets/train_dataset_{dataset}/{filename}"
        seqs = pd.read_csv(seq_path, sep="\t")
        all_v_calls.update(seqs["v_call"].unique())
        all_j_calls.update(seqs["j_call"].unique())

print("Total unique v_calls across all datasets:", len(all_v_calls))
print("Total unique j_calls across all datasets:", len(all_j_calls))

Total unique v_calls across all datasets: 140
Total unique j_calls across all datasets: 28


In [10]:
# Build full vocabulary across all datasets
all_v_calls = sorted(list(all_v_calls))
all_j_calls = sorted(list(all_j_calls))

v_call_to_idx = {v: i for i, v in enumerate(all_v_calls)}
j_call_to_idx = {j: i for i, j in enumerate(all_j_calls)}

print("v_call vocab size:", len(v_call_to_idx))
print("j_call vocab size:", len(j_call_to_idx))
print("\nSample v_calls:", all_v_calls[:5])
print("Sample j_calls:", all_j_calls[:5])

v_call vocab size: 140
j_call vocab size: 28

Sample v_calls: ['TCRBV01-01', 'TCRBV02-01', 'TCRBV03-01/03-02', 'TCRBV04', 'TCRBV04-01']
Sample j_calls: ['TCRBJ01-01', 'TCRBJ01-02', 'TCRBJ01-03', 'TCRBJ01-04', 'TCRBJ01-05']


In [11]:
# Check max junction length across all datasets
max_length = 0

for dataset in range(1, 9):
    meta_path = f"data/train_datasets/train_datasets/train_dataset_{dataset}/metadata.csv"
    meta = pd.read_csv(meta_path)
    
    for filename in meta["filename"].values[:5]:
        seq_path = f"data/train_datasets/train_datasets/train_dataset_{dataset}/{filename}"
        seqs = pd.read_csv(seq_path, sep="\t")
        max_length = max(max_length, seqs["junction_aa"].str.len().max())

print("Max junction length across all datasets:", max_length)

Max junction length across all datasets: 31


In [12]:
# Save vocabulary info
AMINO_ACIDS = sorted(list("ACDEFGHIKLMNPQRSTVWY"))
MAX_JUNCTION_LEN = 31

vocab = {
    "amino_acids": AMINO_ACIDS,
    "aa_to_idx": {aa: i for i, aa in enumerate(AMINO_ACIDS)},
    "v_calls": all_v_calls,
    "v_call_to_idx": v_call_to_idx,
    "j_calls": all_j_calls,
    "j_call_to_idx": j_call_to_idx,
    "max_junction_len": MAX_JUNCTION_LEN,
    "input_dim": MAX_JUNCTION_LEN * len(AMINO_ACIDS) + len(all_v_calls) + len(all_j_calls)
}

print("Input dimension:", vocab["input_dim"])
print("Amino acid vocab:", vocab["amino_acids"])

Input dimension: 788
Amino acid vocab: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
